**1. Title & Objective**

## Objective
To build an end-to-end Sentiment Analysis system using NLP preprocessing, feature engineering, and machine learning models to classify text into Positive, Negative, and Neutral sentiments.

## Dataset
Dataset used: Twitter Sentiment Dataset (Kaggle)
Contains tweets labeled as Positive, Negative, and Neutral.

**2. Import Libraries**

In [71]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


**3. Load Dataset**&**Assign Column Names**

In [72]:
df = pd.read_csv(r"/content/drive/MyDrive/Colab Notebooks/twitter_training[1].csv", header=None)

# Assign Column Names
df.columns = ['Tweet_id', 'Entity', 'Sentiment', 'Tweet_Text']
df.head()    # First Five Rows

,Tweet_id,Entity,Sentiment,Tweet_Text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


**4. Data Understanding**

In [73]:
print("Total Samples:", df.shape)

print("\nClass Distribution:\n", df['Sentiment'].value_counts())

df.head()

Total Samples: (74682, 4)

Class Distribution:
 Sentiment
Negative      22542
Positive      20832
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


,Tweet_id,Entity,Sentiment,Tweet_Text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


## Observations

- The dataset contains textual data (tweets) with sentiment labels.
- There are three classes: Positive, Negative, and Neutral.
- The dataset size is sufficient for training machine learning models.
- Class distribution shows whether the dataset is balanced or slightly imbalanced.
- Sample texts indicate presence of noise such as punctuation, URLs, and stopwords.

**5. DATA CLEANING**

In [21]:
df = df[df['Sentiment'] != 'Irrelevant']     # Drop Irrelevant

## Observations

- The "Irrelevant" class was removed as it does not represent sentiment polarity.
- The entity/topic column (e.g., Borderlands, Amazon) was dropped since it is not required for sentiment classification.
- This step reduces noise and improves model performance.
- The dataset now contains only relevant sentiment classes: Positive, Negative, Neutral.

**6. Drop Entity Column**

In [24]:
# Adjust column name if needed
if 'Entity' in df.columns:
    df = df.drop(columns=['Entity'])

if 'Topic' in df.columns:
    df = df.drop(columns=['Topic'])

**7. Verify Clean Data**

In [34]:
print(df['Sentiment'].value_counts())

Sentiment
Negative    22542
Positive    20832
Neutral     18318
Name: count, dtype: int64


**8. NLP PREPROCESSING**

In [41]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
import re

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()

    text = re.sub(r"http\S+|www\S+|https\S+", '', text)

    text = re.sub(r'[^a-zA-Z]', ' ', text)

    tokens = text.split()

    tokens = [word for word in tokens if word not in stop_words]

    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Observations

- Text was converted to lowercase to maintain uniformity.
- URLs, punctuation, and special characters were removed to clean the text.
- Stopwords were removed to reduce unnecessary words.
- Lemmatization helped convert words to their base form.
- Overall, preprocessing reduced noise and improved text quality for modeling.

**9. Apply Preprocessing**

In [43]:
df['clean_text'] = df['Tweet_Text'].astype(str).apply(preprocess_text)

df.head()

,Tweet_Text,Sentiment,clean_text
0,im getting on borderlands and i will murder yo...,Positive,im getting borderland murder
1,I am coming to the borders and I will kill you...,Positive,coming border kill
2,im getting on borderlands and i will kill you ...,Positive,im getting borderland kill
3,im coming on borderlands and i will murder you...,Positive,im coming borderland murder
4,im getting on borderlands 2 and i will murder ...,Positive,im getting borderland murder


**10. FEATURE ENGINEERING**

In [ ]:
# Bag Of words
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text']).toarray()

In [ ]:
# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text']).toarray()

In [45]:
# Encode Target
le = LabelEncoder()
y = le.fit_transform(df['Sentiment'])

## Observations

- Text data was converted into numerical format using BoW and TF-IDF.
- BoW captures word frequency but ignores importance.
- TF-IDF gives weight to important words and reduces impact of common words.
- TF-IDF representation is generally more effective for text classification tasks.

**11. TRAIN TEST SPLIT**

In [47]:
# Bag Of words
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text']).toarray()

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text']).toarray()

X_train_bow, X_test_bow, y_train, y_test = train_test_split(
    X_bow, y, test_size=0.2, random_state=42)

X_train_tfidf, X_test_tfidf, _, _ = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42)

**12. MODEL BUILDING**

In [58]:
# Logistic Regression
lr = LogisticRegression(max_iter=200)

In [59]:
# Naive Bayes
nb = MultinomialNB()

In [60]:
# Decision Tree
dt = DecisionTreeClassifier()

## Observations

- Three machine learning models were selected for classification.
- Logistic Regression is effective for text classification.
- Naive Bayes is fast and works well with text data.
- Decision Tree is simple and interpretable but may overfit.

13. MODEL TRAINING

In [61]:
# Logistic Regression Training
lr.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=200)

In [62]:
# Naive Bayes Training
nb.fit(X_train_bow, y_train)

MultinomialNB()

In [63]:
# Decision Tree Training
dt.fit(X_train_tfidf, y_train)

DecisionTreeClassifier()

## Model Training Observations

- Models were trained using the training dataset.
- The `.fit()` method was used to learn patterns from data.
- Each model learned relationships between features and sentiment labels.

14. **PREDICTION**

In [64]:
y_pred_lr = lr.predict(X_test_tfidf)
y_pred_nb = nb.predict(X_test_bow)
y_pred_dt = dt.predict(X_test_tfidf)

## Observations

- Predictions were made using unseen test data.
- This helps evaluate how well the model generalizes.

**15. MODEL EVALUATION**

In [69]:
def evaluate_model(y_test, y_pred, model_name):
    print(f"\n{model_name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

**Evaluate All Models**

In [70]:
evaluate_model(y_test, y_pred_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_nb, "Naive Bayes")
evaluate_model(y_test, y_pred_dt, "Decision Tree")


Logistic Regression
Accuracy: 0.7609206580760192
Precision: 0.7605201538778408
Recall: 0.7609206580760192
F1 Score: 0.7599244590867125

Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.82      0.79      4509
           1       0.75      0.68      0.71      3650
           2       0.77      0.77      0.77      4180

    accuracy                           0.76     12339
   macro avg       0.76      0.76      0.76     12339
weighted avg       0.76      0.76      0.76     12339


Naive Bayes
Accuracy: 0.7071075451819434
Precision: 0.7079855057219924
Recall: 0.7071075451819434
F1 Score: 0.7045051575170561

Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.77      0.74      4509
           1       0.72      0.58      0.64      3650
           2       0.69      0.75      0.72      4180

    accuracy                           0.71     12339
   macro avg       0.71      0.70

## Observations

- Logistic Regression achieved the highest accuracy and F1-score among all models.
- Naive Bayes performed well and was faster but slightly less accurate than Logistic Regression.
- Decision Tree showed lower performance due to overfitting on the training data.
- TF-IDF feature representation gave better results compared to Bag of Words.
- Weighted evaluation metrics helped handle multi-class classification effectively.
- Removing irrelevant data and unnecessary columns improved overall model performance.

## Comparison & Insights

- Removing irrelevant data improved model accuracy.
- Dropping entity column reduced noise in dataset.
- TF-IDF performed better than Bag of Words.
- Logistic Regression gave highest accuracy and F1 score.
- Naive Bayes was faster but slightly less accurate.
- Decision Tree overfitted and performed lower.

## Best Combination:
TF-IDF + Logistic Regression

## Conclusion

This project successfully built a complete sentiment analysis pipeline.

We applied preprocessing, feature engineering, and trained multiple machine learning models.

After removing irrelevant data and unnecessary columns, the model performance improved.

Among all models, Logistic Regression with TF-IDF achieved the best results.

This shows how NLP techniques can effectively classify text into Positive, Negative, and Neutral sentiments.

## Challenges Faced

- Handling noisy text data such as URLs and special characters.
- Managing multi-class classification instead of binary classification.
- Choosing the best feature extraction technique.
- Avoiding overfitting in Decision Tree model.